# Differential Privacy Training with AdvSecureNet

This notebook demonstrates how to train a model with differential privacy using AdvSecureNet. We'll train a ResNet-18 model on the CIFAR-10 dataset while preserving privacy using the Opacus library in the background.

## What is Differential Privacy?

Differential privacy limits the extent to which the model’s output can reveal information about any individual training example.

### Key Parameters:
- **`noise_multiplier`**: Controls the amount of noise added during training. Higher values = more privacy but potentially lower utility
- **`max_grad_norm`**: Maximum L2 norm for gradient clipping. Bounds the sensitivity of the model
- **`delta`**: Privacy parameter that bounds the probability of privacy failure
- **`kwargs`**: Additional Opacus parameters ⚠️ **Note**: When using kwargs, ensure that parameter names match exactly what Opacus expects to avoid runtime errors. Compatibility with advsecurenet is not guaranteed for all possible kwargs modifications.

## Step 1: Setup and Imports

In [ ]:
# Core imports
import torch

# AdvSecureNet imports
from advsecurenet.models.model_factory import ModelFactory
from advsecurenet.datasets.dataset_factory import DatasetFactory
from advsecurenet.dataloader.data_loader_factory import DataLoaderFactory
from advsecurenet.trainer.trainer import Trainer
from advsecurenet.shared.types.configs.preprocess_config import (
    PreprocessConfig,
    PreprocessStep,
)
from advsecurenet.shared.types.configs.device_config import DeviceConfig
from advsecurenet.shared.types.configs import TrainConfig
from shared.types.configs.base import (
    CheckpointBase,
    FinalModelBase,
    OptimizationBase,
    DifferentialPrivacyBase
)
from advsecurenet.shared.types.configs.train_config import (
    ModelConfig,
    TrainingProcessConfig,
    DeviceConfig
)

## Step 2: Create ResNet-18 Model

We'll create a ResNet-18 model configured for CIFAR-10 (10 classes).

In [ ]:
# Create ResNet-18 model for CIFAR-10
model = ModelFactory.create_model(
    model_name="resnet18",  
    pretrained=False 
)

# Make the model compatible with Opacus by fixing in-place operations
from opacus.validators import ModuleValidator

# First, fix in-place operations manually
def fix_inplace_operations(module):
    for name, child in module.named_children():
        if isinstance(child, torch.nn.ReLU):
            child.inplace = False
        else:
            fix_inplace_operations(child)

# Apply the manual fix to the underlying model
if hasattr(model, 'model'):
    fix_inplace_operations(model.model)
else:
    fix_inplace_operations(model)

# Then apply Opacus validator fix
model = ModuleValidator.fix(model)

print("Loaded ResNet-18 model")
print(f"Model parameters: {sum(p.numel() for p in model.parameters())}")

## Step 3: Setup Data Preprocessing

Configure preprocessing transforms for CIFAR-10 dataset.

In [ ]:
# Create preprocessing configuration
preprocess_config = PreprocessConfig(
    steps=[
        PreprocessStep(name="Resize", params={"size": 32}),
        PreprocessStep(name="CenterCrop", params={"size": 32}),
        PreprocessStep(name="ToTensor"),
        PreprocessStep(
            name="ToDtype", params={"dtype": "torch.float32", "scale": True}
        ),
        PreprocessStep(
            name="Normalize",
            params={"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
        ),
    ]
)

## Step 4: Create CIFAR-10 Dataset and DataLoader

Load the CIFAR-10 dataset and create data loaders for training.

In [ ]:
# Create CIFAR-10 dataset
dataset = DatasetFactory.load_dataset(
    dataset_name="cifar10", 
    preprocessing=preprocess_config, 
    num_classes=10
)

train_data = dataset['train']
test_data = dataset['test']

print("CIFAR-10 dataset loaded")
print(f"Training samples: {len(train_data):,}".replace(",", "'"))
print(f"Test samples: {len(test_data):,}".replace(",", "'"))

In [ ]:
# Create data loaders
# Note: For differential privacy, batch size should be chosen carefully
# Smaller batches may provide better privacy but slower training
train_loader = DataLoaderFactory.create_dataloader(
    dataset=train_data, 
    batch_size=128,  # Balanced batch size for DP training
    shuffle=True,
    drop_last=True  # Important for DP: ensures consistent batch sizes
)

print("Data loader created")
print(f"Training batches: {len(train_loader)}")
print(f"Training batch size: {train_loader.batch_size}")

## Step 5: Configure Differential Privacy

This is the key step where we configure differential privacy parameters.

### Privacy Parameters Explained:
- **`noise_multiplier=1.2`**: Moderate noise level for reasonable privacy-utility trade-off
- **`max_grad_norm=1.0`**: Standard gradient clipping threshold
- **`delta=1e-5`**: Small probability of privacy failure (< 1 in 100,000)
- **`kwargs`**: Additional Opacus-specific parameters

In [ ]:
# Configure differential privacy
# IMPORTANT: When using kwargs, ensure parameter names match Opacus expectations exactly!
differential_privacy_config = DifferentialPrivacyBase(
    enable=True,
    noise_multiplier=1.2,      # Higher values = more privacy, lower utility
    max_grad_norm=1.0,         # Gradient clipping threshold
    delta=1e-5,                # Privacy failure probability
    kwargs={
        # Example additional parameters (uncomment as needed)
        # "clipping": "flat",     # Gradient clipping method: "flat" or "fast"
        # "loss_reduction": "mean"  # How to reduce loss across samples
    }
)

print("Differential Privacy Configuration:")
print(f"   • Enabled: {differential_privacy_config.enable}")
print(f"   • Noise Multiplier: {differential_privacy_config.noise_multiplier}")
print(f"   • Max Gradient Norm: {differential_privacy_config.max_grad_norm}")
print(f"   • Delta: {differential_privacy_config.delta}")
print(f"   • Additional kwargs: {differential_privacy_config.kwargs}")
print()
print("NOTE: Higher noise_multiplier = more privacy but potentially lower model performance")

## Step 6: Create Training Configuration

Configure the training process with differential privacy enabled.

In [ ]:
# Training configuration with differential privacy
config = TrainConfig(
    model_config=ModelConfig(model=model),
    training_process_config=TrainingProcessConfig(
        train_loader=train_loader,
        epochs=1,                          
        learning_rate=0.01,               
        criterion="cross_entropy",
        verbose=True,
    ),
    optimization_config=OptimizationBase(
        optimizer="sgd",               # SGD often works better with DP
        optimizer_kwargs={
            "momentum": 0.9,
            "weight_decay": 1e-4
        }
    ),
    device_config=DeviceConfig(),      # If DeviceConfig gets initalized empty it uses the first available device in the following priortity order: CUDA -> MPS -> CPU
    checkpoint_config=CheckpointBase(),
    final_model_config=FinalModelBase(
        save_final_model=True,
        save_model_path="./models",
        save_model_name="resnet18_cifar10_dp"
    ),
    differential_privacy_config=differential_privacy_config,
)

print("Training Configuration:")
print(f"   • Epochs: {config.training_process_config.epochs}")
print(f"   • Learning Rate: {config.training_process_config.learning_rate}")
print(f"   • Optimizer: {config.optimization_config.optimizer}")
print(f"   • Differential Privacy: {'ENABLED' if config.differential_privacy_config.enable else 'DISABLED'}")

## Step 7: Initialize Trainer and Start Training

Create the trainer and begin differential privacy training.

In [ ]:
# Create trainer
trainer = Trainer(config)
print(f"Device: {trainer._device}")
print("Trainer initialized with differential privacy!")
print("Starting training with privacy-preserving guarantees...\n")

# Start training
trainer.train()

print()
print("Differential privacy training test completed successfully!")